In [ ]:
import gradio as gr
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors  # 👈 确保这里有 AllChem
from rdkit.Chem.Draw import rdMolDraw2D
from PIL import Image
import io
import cairosvg
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
# 1. 加载模型
device="cuda"
model_name = "merged_model"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="auto", 
    dtype=torch.float16)
model.eval()

In [ ]:
# ==============================
# 2️⃣ 核心推理逻辑 (复用评估脚本)
# ==============================
SYSTEM_PROMPT = """You are an expert in thermodynamics and vapor-liquid equilibrium.
Your task is to predict equilibrium properties of binary mixtures from molecular structures and operating conditions.
Return only the requested numerical results.
"""

def build_messages(user_input, assistant_output=None):
    user_content = SYSTEM_PROMPT.strip() + "\n\n" + user_input.strip()
    messages = [{"role": "user", "content": user_content}]
    if assistant_output is not None:
        messages.append({"role": "assistant", "content": assistant_output})
    return messages

def parse_vle_output(text, task_type="isobaric"):
    text = str(text)
    y1, T, P = None, None, None
    m = re.search(r"Vapor-phase mole fraction of component 1\s*:\s*([-+]?\d*\.?\d+)", text, re.I)
    if m: y1 = float(m.group(1))
    if task_type == "isobaric":
        m = re.search(r"Equilibrium temperature\s*:\s*([-+]?\d*\.?\d+)", text, re.I)
        if m: T = float(m.group(1))
    else:
        m = re.search(r"Equilibrium pressure\s*:\s*([-+]?\d*\.?\d+)", text, re.I)
        if m: P = float(m.group(1))
    return y1, T, P

# ✅ 等压任务专用：拼装标准输入文本
def build_isobaric_input(smiles1, smiles2, pressure, x1):
    return f"""Task: Predict the vapor-liquid equilibrium state of a binary mixture under isobaric conditions.
Given:
Component 1 SMILES: {smiles1}
Component 2 SMILES: {smiles2}
Operating pressure: {pressure:.3f} kPa
Liquid-phase mole fraction of component 1: {x1:.3f}
Please predict:
1. Vapor-phase mole fraction of component 1
2. Equilibrium temperature"""

# ✅ 模型推理抽成独立函数，单点预测和相图扫描共用
def run_model(user_input):
    messages = build_messages(user_input=user_input, assistant_output=None)
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True
    ).to(device)
    input_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.pad_token_id
        outputs = model.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_id
        )
    return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

def draw_phase_diagram(smiles1, smiles2, pressure, n_points=20):
    xs = [round(i / (n_points - 1), 3) for i in range(n_points)]
    x_points, y_points, T_points = [], [], []
    for x in xs:
        try:
            raw = run_model(build_isobaric_input(smiles1, smiles2, pressure, x))
            y1, T, _ = parse_vle_output(raw, "isobaric")
            if T is None:
                continue
            x_points.append(x)
            T_points.append(T)
            y_points.append(y1 if y1 is not None else x)
        except Exception:
            continue
    if not T_points:
        return None

    # ✅ 全局样式：字体更大、线更粗
    plt.rcParams.update({
        "font.size": 12,
        "axes.linewidth": 1.2,
        "figure.facecolor": "white",
    })

    # ✅ 画布加大、dpi 提高 → 清晰不糊
    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

    ax.plot(x_points, T_points, "o-", color="#0000FF",
            linewidth=2, markersize=8, label="Bubble point")
    ax.plot(y_points, T_points, "s-", color="#FF0000",
            linewidth=2, markersize=8, label="Dew point")

    # ✅ 两相区浅绿阴影，专业感拉满
    #ax.fill_betweenx(T_points, x_points, y_points, color="#2ca02c", alpha=0.12)

    ax.set_xlabel("Mole fraction of component 1", fontsize=13, fontweight="bold")
    ax.set_ylabel("Temperature (K)", fontsize=13, fontweight="bold")
    ax.set_title(f"Predicted  T–x–y Diagram ({pressure:.3f} kPa)",
                 fontsize=14, fontweight="bold")
    ax.legend(fontsize=11, loc="upper right")
    ax.grid(alpha=0.3)
    ax.set_xlim(-0.1, 1.1)
    fig.tight_layout()
    return fig      
# ==============================
# 3️⃣ Gradio 预测函数（等压专用，4 入 2 出）
# ==============================
def gradio_predict(smiles1, smiles2, pressure, x1):
    torch.cuda.empty_cache()

    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    if mol1 is None or mol2 is None:
        return "Error: Invalid SMILES input.", None

    # 单点预测
    try:
        raw_pred = run_model(build_isobaric_input(smiles1, smiles2, pressure, x1))
    except Exception as e:
        return f"Model Inference Error: {e}", None

    pred_y1, pred_T, _ = parse_vle_output(raw_pred, "isobaric")

    # ✅ 只输出两行结果
    if pred_y1 is not None and pred_T is not None:
        text = (f"Vapor-phase mole fraction of component 1: {pred_y1:.3f}\n"
                f"Equilibrium temperature: {pred_T:.2f} K")
    else:
        text = raw_pred   # 万一解析失败，兜底显示原始输出

    # 画相图
    try:
        img = draw_phase_diagram(smiles1, smiles2, pressure)
    except Exception:
        img = None

    return text, img

In [ ]:
# ==============================
# 4️⃣ Gradio 界面
# ==============================
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue"),
    css="""
    .main-container { max-width: 900px; margin: 15px auto; padding: 20px; background-color: #fafafa; border-radius: 12px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); }
    .header { text-align: center; font-size: 22px; font-weight: bold; background-color: #e8e8e8; padding: 10px; border-radius: 8px; margin-bottom: 12px; }
    .info { background-color: #f0f7ff; border-left: 4px solid #2196f3; padding: 10px 15px; font-size: 13px; line-height: 1.6; border-radius: 6px; margin-bottom: 15px; }
    textarea { font-weight: bold !important; background-color: #f2f2f2 !important; border: 1px solid #333 !important; border-radius: 8px !important; }
    .gr-button, button { border-radius: 5px !important; font-size: 12px !important; padding: 3px 8px !important; width: 180px !important; min-width: unset !important; max-width: 180px !important; transition: all 0.2s ease-in-out; }
    button.secondary, .gr-button.secondary { background-color: white !important; border: 1px solid #bbb !important; color: #333 !important; font-weight: 500 !important; box-shadow: 0 1px 2px rgba(0,0,0,0.05); }
    button.secondary:hover, .gr-button.secondary:hover { background-color: #f7f7f7 !important; transform: scale(1.02); border-color: #999 !important; }
    #button-row { margin-top: -6px !important; }
    """
) as demo:

    with gr.Column(elem_classes=["main-container"]):
        gr.Markdown("<div class='header'>VLE behavior Prediction AI (Binary Mixture)</div>")
        gr.Markdown("""
        <div class='info'>
         1. This tool predicts the <b>Vapor-Liquid Equilibrium (VLE)</b> behavior of binary mixtures using fine-tuned LLM.<br>
         2. Enter SMILES and operating conditions (isobaric only 101.322 kPa).
        </div>
        """)

        with gr.Row():
            comp1_smiles = gr.Textbox(label="Component 1 SMILES", placeholder="e.g., CC(C)O", scale=1, lines=1.1)
            comp2_smiles = gr.Textbox(label="Component 2 SMILES", placeholder="e.g., CC1=CC=CC=C1", scale=1, lines=1.1)

        with gr.Row():
            x1 = gr.Number(label="Liquid-phase mole fraction of component 1", value=0.5, precision=3, scale=1)
            pressure = gr.Number(label="Operating Pressure (kPa)", value=101.325, precision=3, scale=1)

        with gr.Row(elem_id="button-row"):
            submit_btn = gr.Button("Submit and Predict VLE", elem_classes=["secondary"])

        with gr.Row(equal_height=True):
            with gr.Column(scale=2):
                model_output = gr.Textbox(label="VLE Prediction Output", interactive=False, lines=3.5)
            with gr.Column(scale=4):
                # ✅ 用 gr.Plot 显示 matplotlib 图，自动铺满整列
                phase_plot = gr.Plot(label="Predicted VLE Phase Diagram",show_label = False)

        # ✅ 只保留一份 click 绑定
        submit_btn.click(
            fn=gradio_predict,
            inputs=[comp1_smiles, comp2_smiles, pressure, x1],
            outputs=[model_output, phase_plot]
        )

In [ ]:
demo.close()
demo.launch(server_name="0.0.0.0")